In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score , root_mean_squared_error , mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer
from xgboost import XGBRegressor
import optuna
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice 

In [2]:
dagshub.init(repo_owner='mridul0010', repo_name='NYC-Taxi-Trip-Duration', mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/NYC-Taxi-Trip-Duration"

Repository mridul0010/NYC-Taxi-Trip-Duration initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow")

In [4]:
mlflow.set_experiment("2. HyperParameter Tuning - XGBoost")

<Experiment: artifact_location='mlflow-artifacts:/6bd51d3a00b846af84b31c74436e30b8', creation_time=1784398398432, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1784398398432, lifecycle_stage='active', name='2. HyperParameter Tuning - XGBoost', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
X_train = pd.read_csv("../data/processed/baseline/features.csv")
X_test  = pd.read_csv("../data/processed/baseline/features_test.csv")
y_train = pd.read_csv("../data/processed/baseline/labels.csv")
y_test = pd.read_csv("../data/processed/baseline/labels_test.csv")

In [6]:
print("Shape of X_train :-",X_train.shape)
print("Shape of y_train :-",y_train.shape)
print("Shape of X_test :-",X_test.shape)
print("Shape of y_test :-",y_test.shape)

Shape of X_train :- (1125732, 28)
Shape of y_train :- (1125732, 1)
Shape of X_test :- (281434, 28)
Shape of y_test :- (281434, 1)


In [7]:
pd.set_option('display.max_columns' , None)

In [8]:
def objective(trial):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "XGBoost")
        mlflow.set_tag("model_type", "Regressor")
        
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 800),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15 , log = True),
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
            "min_child_weight": trial.suggest_int("min_child_weight", 10, 50),
            "gamma": trial.suggest_float("gamma", 1e-3, 5.0 , log = True), 
            "reg_lambda": trial.suggest_float("reg_lambda", 1, 20),
            'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 0.85),
            "random_state": 42
        }
        
        xgb = XGBRegressor(**param)
        model = TransformedTargetRegressor(
            regressor= xgb,
            func=np.log1p,
            inverse_func=np.expm1
        )

        
        model.fit(X_train, y_train.squeeze())
        
        cv_score = cross_val_score(
            model ,
            X_train, 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())
        
        mlflow.log_params(param)
        mlflow.log_metric("CV Score", mean_score)
        
        return mean_score

In [9]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=10 , n_jobs=1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the XGB on best param
    best_xgb = XGBRegressor(**study.best_params , n_jobs = -1)

    best_model = TransformedTargetRegressor(
        regressor=best_xgb,
        func=np.log1p,
        inverse_func=np.expm1
    )

    best_model.fit(X_train , y_train.squeeze())

    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    scores = cross_val_score(
        best_model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=1
    )

    # logging metrics
    mlflow.log_metric("Training_error_MAE" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_MAE" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_error_RMSE" ,root_mean_squared_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_RMSE" ,root_mean_squared_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    # log the best model 
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="model_xgb",
        serialization_format="cloudpickle"
    )

[I 2026-07-19 23:23:31,705] A new study created in memory with name: no-name-55a7f812-8de3-4763-a515-a22584ac3f24


  0%|          | 0/10 [00:00<?, ?it/s]

🏃 View run marvelous-horse-868 at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5/runs/ea312b033489427cb7c7c2c51c886339
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5
[I 2026-07-19 23:26:32,753] Trial 0 finished with value: 3.2451001405715942 and parameters: {'n_estimators': 543, 'max_depth': 5, 'learning_rate': 0.02653695800695537, 'subsample': 0.8153773206435087, 'min_child_weight': 30, 'gamma': 0.00587400396697003, 'reg_lambda': 13.46507373583666, 'colsample_bytree': 0.5221878514537319}. Best is trial 0 with value: 3.2451001405715942.
🏃 View run classy-mouse-278 at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5/runs/c03c9de38aa74bee81df80886583e96d
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5
[I 2026-07-19 23:34:24,918] Trial 1 finished with value: 3.0703675746917725 and parameters: {'n_estimators': 701, 'max_depth': 1

2026/07/20 00:00:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5/runs/20e7a40d941946d0b3fcaa63d5eb8c76
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/5


In [10]:
study.best_value

3.05751314163208

In [11]:
study.best_params

{'n_estimators': 552,
 'max_depth': 11,
 'learning_rate': 0.025499777205079798,
 'subsample': 0.6197398900628848,
 'min_child_weight': 31,
 'gamma': 0.00450344407107242,
 'reg_lambda': 6.832878421024383,
 'colsample_bytree': 0.5671066489414062}

In [12]:
best_xgb = XGBRegressor(**study.best_params)

model = TransformedTargetRegressor(
        regressor=best_xgb,
        func=np.log1p,
        inverse_func=np.expm1
    )

model.fit(X_train , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","XGBRegressor(...ree=None, ...)"
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[<U38](28,)","['target_encoded__pickup_zone','target_encoded__dropoff_zone', 'target_encoded__route_time_density',..., 'standard_scaled__is_interstate_trip','standard_scaled__is_late_night', 'standard_scaled__is_weekend_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,28
regressor_ regressor_: objectFitted regressor.,XGBRegressor,"XGBRegressor(...ree=None, ...)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None


In [13]:
y_pred = model.predict(X_test)

r2score = r2_score(y_test , y_pred)
rmse = root_mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)

print("R2 Score :-",r2score)
print("RMSE :-",rmse)
print("MAE :-",mae)

R2 Score :- 0.791357159614563
RMSE :- 4.978411674499512
MAE :- 3.057295322418213


In [14]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [15]:
fig_history.show()

In [16]:
fig_parallel.show()

In [17]:
fig_importance.show()

In [18]:
fig_slice.show()